# 02 - Branched-cover homology and characters

The specialized invariants in later notebooks require more than the abstract isomorphism type of a homology group. They need to know which connected-sum component and satellite layer supplied each coordinate. This notebook explains that structural bookkeeping.

## Learning objectives

- compute `H_1` of a cyclic branched cover;
- distinguish structural factors from canonical invariant factors;
- construct and manipulate homology elements;
- define exact `Q/Z`-valued characters in structural coordinates;
- restrict a character to one satellite layer; and
- convert Smith-generator values to the deck orbit used by metabelian formulas.

## 1. Setup

In [ ]:
from pathlib import Path
from pprint import pprint
import sys

repository_root = Path.cwd()
if repository_root.name == "notebooks":
    repository_root = repository_root.parent
source_directory = repository_root / "src"
if str(source_directory) not in sys.path:
    sys.path.insert(0, str(source_directory))

from sage.all import QQ
from gaknot import (
    BranchedCoverHomology,
    Character,
    GeneralizedAlgebraicKnot,
    torus_character_orbit,
)

## 2. A first branched cover

The double branched cover of the trefoil has first homology `Z/3Z`. The object retains both the group factors and the branch-knot label.

In [ ]:
trefoil = GeneralizedAlgebraicKnot.torus_knot(2, 3)
trefoil_h1 = BranchedCoverHomology(trefoil, 2)

print(trefoil_h1)
print("cover degree:", trefoil_h1.cover_degree)
print("structural factors:", trefoil_h1.invariant_factors)
print("canonical factors:", trefoil_h1.canonical_invariant_factors)
print("Betti number:", trefoil_h1.betti_number)

For a one-generator group there is no visible difference between structural and canonical factors. The distinction appears for connected sums with coprime cyclic orders.

In [ ]:
cinquefoil = GeneralizedAlgebraicKnot.torus_knot(2, 5)
sum_h1 = BranchedCoverHomology(trefoil + cinquefoil, 2)

print(sum_h1)
print("structural cyclic factors:", sum_h1.invariant_factors)
print("canonical invariant factors:", sum_h1.canonical_invariant_factors)

Abstractly, `Z/3Z + Z/5Z` is cyclic of order 15, which explains the canonical factor `[15]`. The structural list `[3,5]` is retained because its first coordinate came from the trefoil and its second from the cinquefoil.

## 3. Inspecting the satellite decomposition

`decomposition` is a deep copy of the full provenance record. Layers are ordered from the outermost satellite pattern inward, which is the opposite direction from the knot's cabling description.

In [ ]:
cable = GeneralizedAlgebraicKnot.iterated_torus_knot(
    [(2, 3), (2, 5)]
)
cable_h1 = BranchedCoverHomology(cable, 2)

print(cable_h1)
pprint(cable_h1.decomposition)

The outer `(2,5)` layer contributes a `Z/5Z` generator. In this cover the inner layer appears in the decomposition but contributes no character coordinates. Keeping an explicit empty layer prevents the remaining coordinates from being assigned to the wrong cabling operation.

## 4. Homology elements

Elements use the same flattened structural order as `all_invariant_factors`. Finite coordinates are automatically reduced modulo their factors.

In [ ]:
generator = trefoil_h1.element(1)
same_generator = trefoil_h1.element([4])
zero = trefoil_h1.zero()

print("generator:", generator)
print("4 reduces to 1 modulo 3:", same_generator == generator)
print("3 times the generator is zero:", 3 * generator == zero)

An element remembers its parent homology group, not merely its coordinate list. This prevents accidental evaluation by a character defined on a different branched cover.

## 5. Characters as exact maps to `Q/Z`

A character is supplied as `[component][layer][coordinate]`. For the trefoil there is one component, one layer, and one cyclic generator. The image `1/3` has order dividing the generator's order, as required.

In [ ]:
trefoil_character = Character(
    trefoil_h1,
    [[[QQ(1) / 3]]],
)

print("stored flat values:", trefoil_character.values)
print("chi(generator):", trefoil_character(generator))
print("chi(2*generator):", trefoil_character(2 * generator))
print("chi(3*generator):", trefoil_character(3 * generator))

Values are normalized modulo one, and evaluation returns the representative in `[0,1)`. Both the homology element and character expose defensive copies of their coordinate lists.

For the iterated knot, the nested empty list below is meaningful: it says that the character has no values on the inner layer rather than omitting that layer.

In [ ]:
cable_character = Character(
    cable_h1,
    [[[QQ(1) / 5], []]],
)

print("outer layer values:", cable_character.restrict_to_layer(0, 0))
print("inner layer values:", cable_character.restrict_to_layer(0, 1))

outer_generator = cable_h1.element([1])
print("character on the outer generator:", cable_character(outer_generator))

## 6. From Smith coordinates to a deck orbit

Yanagida's formulas and the divisible-winding satellite formula use the cyclic orbit `a_0,...,a_(p-1)` rather than the public Smith-basis coordinates. `torus_character_orbit` performs the basis conversion. The associated phase arguments are the exact rationals `a_j/q`.

In [ ]:
orbit = torus_character_orbit(2, 5, [QQ(1) / 5])

print("Smith-generator values:", orbit.generator_values)
print("integer deck orbit:", orbit.a_values)
print("phase arguments:", orbit.phase_arguments)
print("orbit sum is zero modulo q:", sum(orbit.a_values) % orbit.q == 0)

The order `(4,1)` is convention-dependent: it comes from the Smith-basis conversion implemented by the package. It must not be replaced by a guessed ordering, even though several reorderings may look algebraically plausible.

## 7. A metabelian twisted Alexander polynomial

For a positive torus knot `T(p,q)`, the implemented formula requires a character on the `p`-fold cover. The trefoil example meets those hypotheses.

In [ ]:
twisted_alexander = trefoil_character.twisted_alexander_polynomial()
twisted_alexander

This convenience method is intentionally restricted to supported positive, single-summand torus knots. It raises `NotImplementedError` outside that domain rather than returning a polynomial whose basis or representation has not been justified.

## Exercises

1. Construct `H_1` of the double cover of `T(2,7)` and identify a generator.
2. Define the character taking that generator to `2/7`; evaluate it on all seven multiples of the generator.
3. Compare structural and canonical factors for a connected sum of `T(2,3)` and `T(2,7)`.
4. Compute the deck orbit associated with the `2/7` character and check the zero-sum congruence.